In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))

import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS
from scipy.stats import f
from statsmodels.stats.outliers_influence import variance_inflation_factor

from Module import panel_utils
from Module.panel_utils import (
    detect_outlier_regions_iqr,
    friedman_seasonality_test,
    run_panel_unit_root_tests,
    seasonal_adjust_panel_series,
    run_friedman_seasonality_tests,
    ModelResultsAggregator,
    run_panel_regressions,
    run_spec_tests,
    run_panel_model_diagnostics,
    run_panel_regressions_trend,
)


Обработка данных

In [2]:
###############################
    #ЗАГРУЗКА + ПРЕДОБРАБОТКА ДАННЫХ
###############################
df_RF = pd.read_excel('База данных_рег и фед показатели.xlsx', skiprows=1, sheet_name = 'data_RF')
df_reg = pd.read_excel('База данных_рег и фед показатели.xlsx',skiprows=1 , sheet_name = 'data')
pickle_filename = 'region_cluster_dataset.pkl'
with open(pickle_filename, 'rb') as f:
    df_region_cluster = pickle.load(f)
pickle_filename = 'mon_shock_dataset.pkl'
with open(pickle_filename, 'rb') as f:
    df_shocks = pickle.load(f)

In [3]:
# Преобразование даты 
df_RF['Date'] = pd.to_datetime(df_RF['Date'])
df_RF = df_RF.sort_values('Date')
df_reg['Date'] = pd.to_datetime(df_reg['Date'])
df_reg = df_reg.sort_values(['Num_reg','Date'])

In [ ]:
###############################
# АНАЛИЗ ПАНЕЛЬНЫХ ДАННЫХ
###############################

# Сортировка
df_reg = df_reg.sort_values(['Region', 'Date']).copy()

#  ДОБАВЛЯЕМ КЛАСТЕРЫ 
df_reg = df_reg.merge(df_region_cluster, on='Region', how='left')

# ============ ФЕДЕРАЛЬНЫЕ ПЕРЕМЕННЫЕ ============
df_reg = df_reg.merge(df_shocks[['Date', 'Mon_Shock']], on='Date', how='left')

df_rf_extra = df_RF[['Date','REER', 'MIACR', 'Covid_dum', 'Sank_dum', 'ROISFIX', 'MaP_Tight_Announcement', 
                     'MaP_Ease_Announcement', 'MaP_Tight_Fact', 'MaP_Ease_Fact',
                     'Oil_p']].copy()
df_reg = df_reg.merge(df_rf_extra, on='Date', how='left')
df_reg.rename(columns={'exc_rate': 'Exc_rate'}, inplace=True)

map_cols = [
    'MaP_Tight_Announcement',
    'MaP_Ease_Announcement',
    'MaP_Tight_Fact',
    'MaP_Ease_Fact'
]
df_reg[map_cols] = df_reg[map_cols].fillna(0)

#Расчетные показатели
df_reg['Oil_p_share_mining'] = df_reg['Oil_p'] * df_reg['Share_mining']
df_reg['Share_Mort_Progr_new'] = df_reg['New_Loans_Progr'] / df_reg['New_Loans_Mort']

# Создаем лаги
df_reg['D_top5_rozn_lag1'] = df_reg.groupby('Region')['D_top5_rozn'].shift(1)
df_reg['CPI_lag1'] = df_reg.groupby('Region')['CPI'].shift(1)
df_reg['Def_Zadolg_Fl_lag1'] = df_reg.groupby('Region')['Def_Zadolg_Fl'].shift(1)
df_reg['Def_Zadolg_Mort_lag1'] = df_reg.groupby('Region')['Def_Zadolg_Mort'].shift(1)
df_reg['Def_Zadolg_ConsCred_lag1'] = df_reg.groupby('Region')['Def_Zadolg_ConsCred'].shift(1)
df_reg['Cap_to_assets_lag1'] = df_reg.groupby('Region')['Cap_to_assets'].shift(1)
        
# Разность монетарного шока
df_reg['d_Mon_Shock'] = df_reg.groupby('Region')['Mon_Shock'].diff()

# Разделим Mon_Shock на позитивный и негативный
df_reg['d_Mon_Shock_neg'] = df_reg['d_Mon_Shock'].where(df_reg['d_Mon_Shock'] < 0, 0)
df_reg['d_Mon_Shock_pos'] = df_reg['d_Mon_Shock'].where(df_reg['d_Mon_Shock'] > 0, 0)

# СПИСОК ПЕРЕМЕННЫХ
initial_vars = [
    #Блок 1 - Зависимые переменные + лаги
    'New_Loans_Fl',
    'New_Loans_Mort',
    'New_Loans_ConsCred',
    #Блок 2 - Переменные интереса + лаги
    'MaP_Tight_Announcement',
    'MaP_Ease_Announcement',
    'MaP_Tight_Fact',
    'MaP_Ease_Fact',
    #Блок 3 - характеристики фин системы
    'D_top5_rozn_lag1',
    'Fin_Dostup',
    'Def_Zadolg_Fl_lag1',
    'Def_Zadolg_Mort_lag1',
    'Def_Zadolg_ConsCred_lag1',
    'Cap_to_assets_lag1',
    #Блок 4 - макроэкономический
    'CPI_lag1',
    'Oil_p_share_mining',
    'REER',
    #Блок 5 - монетарный 
    'd_Mon_Shock_pos',
    'd_Mon_Shock_neg',
    'ROISFIX',
    'MIACR',
    #Блок 6 - кластеры
    'Cluster_1',
    'Cluster_2',
    #Блок 7 - шоки
    'Covid_dum',
    'Sank_dum',
]

df_reg_analys = df_reg[['Region', 'Date'] + initial_vars].copy()
df_reg_analys = df_reg_analys[df_reg_analys['Region'] != 'Москва']
# cols_for_drop = [c for c in df_reg_analys.columns]
# df_reg_analys = df_reg_analys.dropna(subset=cols_for_drop)

IQR очистка

In [5]:
print("\n" + "="*70)
print("ОЧИСТКА РЕГИОНОВ ПО МЕТОДУ IQR")
print("="*70)

key_vars = [
    'New_Loans_Fl',
    'New_Loans_Mort',
    'New_Loans_ConsCred',
    'D_top5_rozn_lag1',
    'Fin_Dostup',
    'Def_Zadolg_Fl_lag1',
    'Def_Zadolg_Mort_lag1',
    'Def_Zadolg_ConsCred_lag1',
    'Cap_to_assets_lag1',
    'CPI_lag1',
    'Oil_p_share_mining',
]
region_quality_iqr = detect_outlier_regions_iqr(df_reg_analys, key_vars, min_obs=60, max_na_pct=0.3)

print(f"\n АНАЛИЗ {len(region_quality_iqr)} РЕГИОНОВ (IQR)")
print(region_quality_iqr.to_string(index=False))

# СТАТИСТИКА
exclude_count = len(region_quality_iqr[region_quality_iqr['Status'] == 'EXCLUDE'])
keep_count = len(region_quality_iqr) - exclude_count

print(f"\n СТАТИСТИКА:")
print(f"Всего регионов: {len(region_quality_iqr)}")
print(f"Исключено: {exclude_count} ({exclude_count/len(region_quality_iqr)*100:.1f}%)")
print(f"Оставлено: {keep_count}")

# СПИСОК ХОРОШИХ РЕГИОНОВ
good_regions_iqr = region_quality_iqr[region_quality_iqr['Status'] == 'KEEP']['Region'].tolist()

# ФИЛЬТРАЦИЯ ДАННЫХ
df_reg_analys= df_reg_analys[df_reg_analys['Region'].isin(good_regions_iqr)].copy()
print(f"\n Форма после очистки (IQR): {df_reg_analys.shape}")


ОЧИСТКА РЕГИОНОВ ПО МЕТОДУ IQR

 АНАЛИЗ 82 РЕГИОНОВ (IQR)
                             Region  N_obs NA_pct  IQR_outliers IQR_pct                       Reasons  Status
                     Алтайский край     80   1.2%             2    2.5%                                  KEEP
                   Амурская область     80   1.2%             0    0.0%                                  KEEP
              Архангельская область     80   1.2%             4    5.0%                                  KEEP
               Астраханская область     80   1.2%             0    0.0%                                  KEEP
               Белгородская область     80   1.2%             1    1.2%                                  KEEP
                   Брянская область     80   1.2%             0    0.0%                                  KEEP
               Владимирская область     80   1.2%             0    0.0%                                  KEEP
              Волгоградская область     80   1.2%            

Преобразование данных

In [6]:
###############################
# Переход к первым разностям и логарифмам
###############################
df_reg = df_reg_analys.copy()

# ============ ЛОГАРИФМИРОВАНИЕ ============

loan_vars = ['New_Loans_Fl', 'New_Loans_Mort', 'New_Loans_ConsCred', 'Fin_Dostup']

for var in loan_vars:
    if var in df_reg.columns:
        df_reg[f'ln_{var}'] = np.log(df_reg[var])
        
ln_loan_vars = [f'ln_{var}' for var in loan_vars if f'ln_{var}' in df_reg.columns]

# ============ СОЗДАНИЕ ПЕРВЫХ РАЗНОСТЕЙ  ============

exclude_cols = ['Region', 'Date']

# Разности для логарифмов 
for var in ln_loan_vars:
    d_var = f'd_{var}'
    df_reg[d_var] = df_reg.groupby('Region')[var].diff()

# Создаем лаги
df_reg['d_ln_New_Loans_Fl_lag1'] = df_reg.groupby('Region')['d_ln_New_Loans_Fl'].shift(1)
df_reg['ln_New_Loans_Fl_lag1'] = df_reg.groupby('Region')['ln_New_Loans_Fl'].shift(1)
df_reg['ln_New_Loans_Mort_lag1'] = df_reg.groupby('Region')['ln_New_Loans_Mort'].shift(1)
df_reg['ln_New_Loans_ConsCred_lag1'] = df_reg.groupby('Region')['ln_New_Loans_ConsCred'].shift(1)
df_reg['MaP_Tight_Announcement_lag1'] = df_reg.groupby('Region')['MaP_Tight_Announcement'].shift(1)
df_reg['MaP_Ease_Announcement_lag1'] = df_reg.groupby('Region')['MaP_Ease_Announcement'].shift(1)
df_reg['MaP_Tight_Fact_lag1'] = df_reg.groupby('Region')['MaP_Tight_Fact'].shift(1)
df_reg['MaP_Ease_Fact_lag1'] = df_reg.groupby('Region')['MaP_Ease_Fact'].shift(1)
df_reg['d_Mon_Shock_pos_lag1'] = df_reg.groupby('Region')['d_Mon_Shock_pos'].shift(1)
df_reg['d_Mon_Shock_neg_lag1'] = df_reg.groupby('Region')['d_Mon_Shock_neg'].shift(1)

# Разделим MIACR на позитивный и негативный
df_reg['d_MIACR'] = df_reg.groupby('Region')['MIACR'].diff()
df_reg['d_MIACR_neg'] = df_reg['d_MIACR'].where(df_reg['d_MIACR'] < 0, 0)
df_reg['d_MIACR_neg_lag1'] = df_reg.groupby('Region')['d_MIACR_neg'].shift(1)
df_reg['d_MIACR_pos'] = df_reg['d_MIACR'].where(df_reg['d_MIACR'] > 0, 0)
df_reg['d_MIACR_pos_lag1'] = df_reg.groupby('Region')['d_MIACR_pos'].shift(1)

# Разделим ROISFIX на позитивный и негативный
df_reg['d_ROISFIX'] = df_reg.groupby('Region')['ROISFIX'].diff()
df_reg['d_ROISFIX_neg'] = df_reg['d_ROISFIX'].where(df_reg['d_ROISFIX'] < 0, 0)
df_reg['d_ROISFIX_neg_lag1'] = df_reg.groupby('Region')['d_ROISFIX_neg'].shift(1)
df_reg['d_ROISFIX_pos'] = df_reg['d_ROISFIX'].where(df_reg['d_ROISFIX'] > 0, 0)
df_reg['d_ROISFIX_pos_lag1'] = df_reg.groupby('Region')['d_ROISFIX_pos'].shift(1)


# Создаем взаимодействия
map_vars = [
    'MaP_Tight_Announcement',
    'MaP_Ease_Announcement',
    'MaP_Tight_Fact',
    'MaP_Ease_Fact',
]

z_vars = [
    'D_top5_rozn_lag1',
    'Cap_to_assets_lag1',
    'Cluster_1',
    'Cluster_2'
]
for m in map_vars:
    for z in z_vars:
        col_name = f"{m}_{z}"
        df_reg[col_name] = df_reg[m] * df_reg[z]

exog_vars = [
    #Блок 1 - Зависимые переменные + лаги
    'ln_New_Loans_Fl',
    'd_ln_New_Loans_Fl',
    'ln_New_Loans_Mort',
    'ln_New_Loans_ConsCred',
    'ln_New_Loans_Fl_lag1',
    'd_ln_New_Loans_Fl_lag1',
    'ln_New_Loans_Mort_lag1',
    'ln_New_Loans_ConsCred_lag1',
    #Блок 2 - Переменные интереса + лаги
    'MaP_Tight_Announcement',
    'MaP_Ease_Announcement',
    'MaP_Tight_Fact',
    'MaP_Ease_Fact',
    'MaP_Tight_Announcement_lag1',
    'MaP_Ease_Announcement_lag1',
    'MaP_Tight_Fact_lag1',
    'MaP_Ease_Fact_lag1',
    #Блок 3 - характеристики фин системы
    'D_top5_rozn_lag1',
    'ln_Fin_Dostup',
    'Def_Zadolg_Fl_lag1',
    'Def_Zadolg_Mort_lag1',
    'Def_Zadolg_ConsCred_lag1',
    'Cap_to_assets_lag1',
    #Блок 4 - макроэкономический
    'CPI_lag1',
    'Oil_p_share_mining',
    'REER',
    #Блок 5 - монетарный 
    'd_Mon_Shock_pos',
    'd_Mon_Shock_neg',
    'd_Mon_Shock_pos_lag1',
    'd_Mon_Shock_neg_lag1',
    'd_ROISFIX_neg',
    'd_ROISFIX_neg_lag1',
    'd_ROISFIX_pos',
    'd_ROISFIX_pos_lag1',
    'd_MIACR_neg',
    'd_MIACR_neg_lag1',
    'd_MIACR_pos',
    'd_MIACR_pos_lag1',
    #Блок 6 - кластеры
    'Cluster_1',
    'Cluster_2',
    #Блок 7 - шоки
    'Covid_dum',
    'Sank_dum',
    #Блок 8 - взаимодействия
    'MaP_Tight_Announcement_D_top5_rozn_lag1',
    'MaP_Tight_Fact_D_top5_rozn_lag1',
    'MaP_Ease_Announcement_D_top5_rozn_lag1',
    'MaP_Ease_Fact_D_top5_rozn_lag1',
    'MaP_Tight_Announcement_Cap_to_assets_lag1',
    'MaP_Tight_Fact_Cap_to_assets_lag1',
    'MaP_Ease_Announcement_Cap_to_assets_lag1',
    'MaP_Ease_Fact_Cap_to_assets_lag1',
    'MaP_Tight_Announcement_Cluster_1',
    'MaP_Tight_Fact_Cluster_1',
    'MaP_Ease_Announcement_Cluster_1',
    'MaP_Ease_Fact_Cluster_1',
    'MaP_Tight_Announcement_Cluster_2',
    'MaP_Tight_Fact_Cluster_2',
    'MaP_Ease_Announcement_Cluster_2',
    'MaP_Ease_Fact_Cluster_2' 
]

# Проверяем, какие переменные реально существуют
exog_vars_existing = [var for var in exog_vars if var in df_reg.columns]
exog_vars_missing = [var for var in exog_vars if var not in df_reg.columns]

df_exog = df_reg[['Region', 'Date'] + exog_vars_existing].copy()
# df_exog = df_exog.dropna()

In [7]:
###############################
# Создание аргументов модели
###############################

dependent_var = 'd_ln_New_Loans_Fl'

exog_vars_base = [
    #Блок 1 - Лаги зависимых переменных
    'd_ln_New_Loans_Fl_lag1',
    #Блок 2 - Переменные интереса + лаги
    # 'MaP_Tight_Fact',
    # 'MaP_Ease_Fact',
    'MaP_Tight_Announcement',
    'MaP_Ease_Announcement',
    #Блок 3 - характеристики фин системы
    'ln_Fin_Dostup',
    'Def_Zadolg_Fl_lag1',
    'Cap_to_assets_lag1',
    #Блок 4 - макроэкономический
    'CPI_lag1',
    'Oil_p_share_mining',
    'REER',
    #Блок 5 - монетарный 
    # 'd_Mon_Shock_pos',
    # 'd_Mon_Shock_neg',
    #Блок 6 - кластеры
    'Cluster_1',
    'Cluster_2',
    #Блок 7 - шоки
    'Covid_dum',
    'Sank_dum'
]

Анализ мультиколлинеарности

In [8]:
###############################
    #VIF-анализ - начальные панельные данные спецификация 1
###############################


vif_variables = [item for item in exog_vars_base if item not in ['x']]
df_vif = df_reg[vif_variables].copy()
df_vif = df_vif.dropna()

X_vif = sm.add_constant(df_vif[vif_variables])

# Расчет VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = vif_variables
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i+1) for i in range(len(vif_variables))]
vif_data = vif_data.sort_values('VIF', ascending=False)

print("\n" + "="*60)
print("РЕЗУЛЬТАТЫ VIF АНАЛИЗА - НАЧАЛЬНЫЕ ПАНЕЛЬНЫЕ ДАННЫЕ СПЕЦИФИКАЦИЯ 1")
print("="*60)
print(vif_data.to_string(index=False))


РЕЗУЛЬТАТЫ VIF АНАЛИЗА - НАЧАЛЬНЫЕ ПАНЕЛЬНЫЕ ДАННЫЕ СПЕЦИФИКАЦИЯ 1
              Variable      VIF
              Sank_dum 2.748151
              CPI_lag1 1.919329
             Covid_dum 1.747548
         ln_Fin_Dostup 1.481351
    Def_Zadolg_Fl_lag1 1.362061
    Cap_to_assets_lag1 1.305204
             Cluster_2 1.231521
                  REER 1.222007
             Cluster_1 1.204310
    Oil_p_share_mining 1.198609
 MaP_Ease_Announcement 1.125412
MaP_Tight_Announcement 1.085694
d_ln_New_Loans_Fl_lag1 1.040363


In [9]:
###############################
# Построение линейных моделей на панельных данных - общая выборка
# (ln_New_Loans_Fl зависимая переменная)
###############################

dependent_var = dependent_var
exog_vars_initial = exog_vars_base

df_regress = df_exog.dropna().copy()
y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_exog, df_exog, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_exog'
)

ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (df_exog)
Зависимая переменная: d_ln_New_Loans_Fl

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:      d_ln_New_Loans_Fl   R-squared:                        0.1791
Estimator:                  PooledOLS   R-squared (Between):             -18.019
No. Observations:                5846   R-squared (Within):               0.1822
Date:                Mon, Jan 26 2026   R-squared (Overall):              0.1791
Time:                        15:25:26   Log-likelihood                    2548.9
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      97.845
Entities:                          74   P-value                           0.0000
Avg Obs:                       79.000   Distribution:                 F(13,5832)
Min Obs:                       79.000                                           
M

c:\PyProjects\CreditResearch\Monetary-policy-and-retail-landing-in-regions\panel_utils.py:1139: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

Cluster_1, Cluster_2

  fe_res = fe_mod.fit(cov_type=cov_type, cluster_entity=cluster_entity)


In [ ]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)

In [ ]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====

run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)

In [ ]:
###############################
    #VIF-анализ - начальные панельные данные спецификация 1
###############################

vif_variables = [
    #Блок 1 - Лаги зависимых переменных
    'ln_New_Loans_Fl',
    'ln_New_Loans_Mort',
    'ln_New_Loans_ConsCred',
    'ln_New_Loans_Fl_lag1',
    'ln_New_Loans_Mort_lag1',
    'ln_New_Loans_ConsCred_lag1',
    #Блок 2 - Переменные интереса + лаги
    'MaP_Tight_Announcement',
    'MaP_Ease_Announcement',
    'MaP_Tight_Fact',
    'MaP_Ease_Fact',
    'MaP_Tight_Announcement_lag1',
    'MaP_Ease_Announcement_lag1',
    'MaP_Tight_Fact_lag1',
    'MaP_Ease_Fact_lag1',
    #Блок 3 - характеристики фин системы
    'D_top5_rozn_lag1',
    'ln_Fin_Dostup',
    'Def_Zadolg_Fl_lag1',
    'Def_Zadolg_Mort_lag1',
    'Def_Zadolg_ConsCred_lag1',
    'Cap_to_assets_lag1',
    #Блок 4 - макроэкономический
    'CPI_lag1',
    'Oil_p_share_mining',
    'REER',
    #Блок 5 - монетарный 
    'Mon_Shock_pos',
    'Mon_Shock_neg',
    'ROISFIX',
    'MIACR',
    'Mon_Shock_pos_lag1',
    'Mon_Shock_neg_lag1',
    'ROISFIX_lag1',
    'MIACR_lag1',
    #Блок 6 - кластеры
    'Cluster_1',
    'Cluster_2',
    #Блок 7 - шоки
    'Covid_dum',
    'Sank_dum',
    #Блок 8 - взаимодействия
    'MaP_Tight_Announcement_D_top5_rozn_lag1',
    'MaP_Tight_Fact_D_top5_rozn_lag1',
    'MaP_Ease_Announcement_D_top5_rozn_lag1',
    'MaP_Ease_Fact_D_top5_rozn_lag1',
    'MaP_Tight_Announcement_Cap_to_assets_lag1',
    'MaP_Tight_Fact_Cap_to_assets_lag1',
    'MaP_Ease_Announcement_Cap_to_assets_lag1',
    'MaP_Ease_Fact_Cap_to_assets_lag1',
    'MaP_Tight_Announcement_Cluster_1',
    'MaP_Tight_Fact_Cluster_1',
    'MaP_Ease_Announcement_Cluster_1',
    'MaP_Ease_Fact_Cluster_1',
    'MaP_Tight_Announcement_Cluster_2',
    'MaP_Tight_Fact_Cluster_2',
    'MaP_Ease_Announcement_Cluster_2',
    'MaP_Ease_Fact_Cluster_2' 
]

df_vif = df_exog[vif_variables].copy()
df_vif = df_vif.dropna()

X_vif = sm.add_constant(df_vif[vif_variables])

# Расчет VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = vif_variables
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i+1) for i in range(len(vif_variables))]
vif_data = vif_data.sort_values('VIF', ascending=False)

print("\n" + "="*60)
print("РЕЗУЛЬТАТЫ VIF АНАЛИЗА - НАЧАЛЬНЫЕ ПАНЕЛЬНЫЕ ДАННЫЕ СПЕЦИФИКАЦИЯ 1")
print("="*60)
print(vif_data.to_string(index=False))